# Experiment 1: grid search and clustering stability

This notebook searches graph hyperparameters using a saved core representation. It never rebuilds the biological input matrices and cannot silently use a different gene order because `load_core_run` validates the manifest.


In [ ]:
from pathlib import Path
import sys


def find_pipeline_root():
    cwd = Path.cwd().resolve()
    candidates = [cwd, cwd / "Cross Species Pipeline", cwd.parent, cwd.parent.parent]
    for candidate in candidates:
        if (candidate / "src" / "cross_species").is_dir():
            return candidate
    raise FileNotFoundError("Could not locate Cross Species Pipeline/src/cross_species")


PIPELINE_ROOT = find_pipeline_root()
PROJECT_ROOT = PIPELINE_ROOT.parent
SRC_ROOT = PIPELINE_ROOT / "src"
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))
RESULTS_ROOT = PIPELINE_ROOT / "results"
DATA_ROOT = PROJECT_ROOT / "data"
print(f"Pipeline root: {PIPELINE_ROOT}")


In [ ]:
from datetime import datetime
from itertools import product
import gc
import anndata as ad
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from cross_species import (
    PARAMS,
    adjacent_agreement,
    cache_neighbors,
    cached_graph,
    evaluate_clusterings,
    load_core_run,
    review_hyperparameter_experiment,
    screen,
    stability,
)


In [ ]:
from cross_species import load_core_run

CORE_RUN_DIR = None  # Set explicitly, or leave None to use the newest core_* run.
if CORE_RUN_DIR is None:
    candidates = [path for path in (RESULTS_ROOT / "core").glob("core_*") if path.is_dir()]
    if not candidates:
        raise FileNotFoundError("No core run found. Run 01_full_pipeline.ipynb first.")
    CORE_RUN_DIR = max(candidates, key=lambda path: path.stat().st_mtime)
core = load_core_run(CORE_RUN_DIR)
prof = core["prof"]
C = core["C"]
graph = core["graph"]
labels = core["labels"]
display(pd.Series(core["manifest"], name="value"))


## Configure and run the 81-setting grid


In [ ]:
RUN_GRID_SEARCH = False
GRID = {
    "k_intra": [25, 50, 100],
    "k_inter": [10, 25, 50],
    "cutoff_intra": [0.85, 0.90, 0.95],
    "cutoff_inter": [0.65, 0.70, 0.75],
}
GRID_OUTPUT_DIR = RESULTS_ROOT / "grid_search" / datetime.now().strftime("grid_%Y%m%d_%H%M%S")

neighbor_cache = cache_neighbors(
    C,
    prof.meta["width_a"],
    max_intra=max(GRID["k_intra"]),
    max_inter=max(GRID["k_inter"]),
)

if RUN_GRID_SEARCH:
    results = screen(
        neighbor_cache, prof.genes, prof.species, GRID, GRID_OUTPUT_DIR,
        resolution=1.0, seed=0,
    )
    agreement = adjacent_agreement(results, GRID_OUTPUT_DIR)
    agreement.to_csv(GRID_OUTPUT_DIR / "adjacent_agreement.csv", index=False)
    display(results)
else:
    print("Set RUN_GRID_SEARCH = True to run the 81-setting screen.")


## Optional seed and resolution stability


In [ ]:
SHORTLIST_IDS = []
if RUN_GRID_SEARCH and SHORTLIST_IDS:
    shortlist = results[results.run_id.isin(SHORTLIST_IDS)]
    stability_output = GRID_OUTPUT_DIR / "stability"
    stability_results = stability(
        neighbor_cache,
        prof.genes,
        prof.species,
        shortlist,
        stability_output,
        seeds=range(5),
        resolutions=(0.5, 1.0, 1.5),
    )
    display(stability_results)
else:
    print("Fill SHORTLIST_IDS after reviewing the grid if stability runs are needed.")


## Optional full module evaluation in batches


In [ ]:
RUN_FULL_BATCHED_EVALUATION = False
BATCH_SIZE = 3

if RUN_FULL_BATCHED_EVALUATION:
    sm_adata_human = ad.read_h5ad(
        DATA_ROOT / "sm_adata/human_GSM9629357_CS23_E2S1_bin50_sm.h5ad"
    )
    sm_adata_mouse = ad.read_h5ad(
        DATA_ROOT / "sm_adata/mouse_E1S1_165_bin50_sm.h5ad"
    )
    parameter_sets = [
        dict(zip(PARAMS, values))
        for values in product(*(GRID[name] for name in PARAMS))
    ]
    experiment_dir = RESULTS_ROOT / "evaluation" / datetime.now().strftime(
        "module_evaluation_%Y%m%d_%H%M%S"
    )
    experiment_dir.mkdir(parents=True, exist_ok=False)
    batch_summaries = []
    for batch_number, start in enumerate(range(0, len(parameter_sets), BATCH_SIZE)):
        clustering_batch = []
        for run_id, parameters in enumerate(parameter_sets[start:start+BATCH_SIZE], start=start):
            candidate_graph = cached_graph(
                neighbor_cache, prof.genes, prof.species, **parameters
            )
            clustering_batch.append({
                "name": f"run_{run_id:03d}",
                **parameters,
                "G": candidate_graph,
                "labels": candidate_graph.leiden(resolution=1.0, seed=0, verbose=False),
            })
        batch_dir = experiment_dir / f"batch_{batch_number:02d}"
        batch_result = evaluate_clusterings(
            clustering_batch,
            C,
            prof,
            expression_by_species={"mouse": sm_adata_mouse, "human": sm_adata_human},
            include_pairwise=False,
            make_plots=True,
            show_plots=False,
            output_dir=batch_dir,
            return_figures=False,
        )
        batch_summary = batch_result["summary"].reset_index()
        batch_summary["batch"] = batch_number
        batch_summaries.append(batch_summary)
        plt.close("all")
        del batch_result, clustering_batch
        gc.collect()
        print(f"Completed batch {batch_number + 1}")
    pd.concat(batch_summaries, ignore_index=True).to_csv(
        experiment_dir / "all_settings_summary.csv", index=False
    )
    print(f"Full evaluation written to: {experiment_dir}")
else:
    print("Set RUN_FULL_BATCHED_EVALUATION = True to evaluate all settings.")


## Review a completed full evaluation


In [ ]:
RUN_HYPERPARAMETER_REVIEW = False
MODULE_EVALUATION_DIR = None  # Set to a module_evaluation_* directory.
if RUN_HYPERPARAMETER_REVIEW:
    review = review_hyperparameter_experiment(
        MODULE_EVALUATION_DIR,
        shortlist_size=8,
        show_plots=True,
    )
    display(review["shortlist"])
else:
    print("Set MODULE_EVALUATION_DIR and RUN_HYPERPARAMETER_REVIEW = True when ready.")
